In [1]:
import cv2
cap = cv2.VideoCapture('/Users/elliekogan/Desktop/videos/133.1_VideoD6Scan.avi')
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Video Resolution: {width}x{height}")
cap.release()

Video Resolution: 0x0


OpenCV: Couldn't read video stream from file "/Users/elliekogan/Desktop/videos/133.1_VideoD6Scan.avi"


In [2]:
import os 
import pandas as pd 
import numpy as np

In [3]:
nose_x = 'nose'
nose_y = 'nose.1'
tail_base_x = 'tail_base'
tail_base_y = 'tail_base.1'

In [4]:
# Define the path to the folder containing your files
folder_path = '/Users/elliekogan/Desktop/videos/csv'
file_path = '/Users/elliekogan/Desktop/videos/csv'
# Read the CSV file
velocity_folder = '/Users/elliekogan/Desktop/videos/csv'

# List all files in the folder
file_list = os.listdir(folder_path)
print(file_list)

['363.2_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '.DS_Store', '364.3_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '365.3_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '363.1_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '363.5_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '365.4_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', 'Velocity', 'Bouts', '363.4_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '364.1_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '365.1_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '363.3_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '364.2_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv', '365.2_VideoDLC_HrnetW32_DLC_vids_2keyFeb

In [5]:
import pandas as pd
# binning for D3
# bin_edges = [0, 169, 187, 206, 225, 300, 318, 375]
# bin_names = ['Habituation', 'CU', 'ITI', 'CU', 'ITI', 'CU', 'ITI']

# binning for D4
bin_edges = [0, 169, 187, 262, 280, 299, 318, 375]
bin_names = ['Habituation', 'CS1', 'ITI1', 'CS2', 'ITI2', 'CS3', 'Post-CS']

# binning for D5
# bin_edges = [0, 169, 187, ... ]
# bin_names = ['Habituation', ... , 'Post-CS']

scanning_duration_df = pd.DataFrame(columns=['Number'] + bin_names)
freezing_duration_df = pd.DataFrame(columns=['Number'] + bin_names)
rearing_duration_df = pd.DataFrame(columns=['Number'] + bin_names)

In [6]:
import pandas as pd

def trim(dlc):
    # Removing the likelihood columns (based on your provided column indices)
    columns_to_remove = [3, 6]  # Adjust if necessary
    dlc = dlc.drop(dlc.columns[columns_to_remove], axis=1)
    dlc['Frame'] = dlc.index

    print("Columns after removal:", dlc.columns)
    print("Number of columns after removal:", dlc.shape[1])

    # Ensure the correct columns are selected
    expected_columns = ['Frame', 'Nose', 'Nose.1', 'Tail_base', 'Tail_base.1']  # Adjust if needed

    # Keep only the expected columns
    dlc = dlc[expected_columns]

    # Remove the first row (if necessary)
    dlc = dlc.iloc[1:]

    # Convert columns to numeric values
    dlc = dlc.apply(pd.to_numeric, errors='coerce')

    # Add a 'Second' column by assuming 'Frame' corresponds to frames per second (8 fps as per your code)
    dlc["Second"] = dlc["Frame"] // 8  # Adjust for frame rate if needed

    # Average coordinates for each second
    result = dlc.groupby("Second").agg({
        'Nose': 'mean', 'Nose.1': 'mean',
        'Tail_base': 'mean', 'Tail_base.1': 'mean',
        }).reset_index()

    return result

In [7]:
def calculate_speed(result):
    # Calculate speed for tailbase
    for i in range(1, len(result)):
        prev_row = result.iloc[i - 1]
        cur_row = result.iloc[i]

        # Calculate speed for Tailbase (same as you already had)
        tailbase_speed = np.sqrt((cur_row['Tail_base'] - prev_row['Tail_base'])**2 +
                                 (cur_row['Tail_base.1'] - prev_row['Tail_base.1'])**2)
        result.at[i, 'Tailbase_Speed'] = tailbase_speed

        # Calculate speed for Nose (new code for nose speed)
        nose_speed = np.sqrt((cur_row['Nose'] - prev_row['Nose'])**2 + (cur_row['Nose.1'] - prev_row['Nose.1'])**2)
        result.at[i, 'Nose_Speed'] = nose_speed

    # Perform similar speed calculations for other body parts if needed

    return result


In [8]:
def scanning_dur(bin_data, bin_end):
    scanning_count = 0
    is_scanning = False

    # Iterate over rows within the bin
    for index, row in bin_data.iterrows():
        nose_speed = row['Nose_Speed']
        tailbase_speed = row['Tailbase_Speed']

        # Check if scanning behavior is detected
        if nose_speed > 10 and tailbase_speed < 10:
            if not is_scanning:  # If scanning just started (first time entering scanning)
                is_scanning = True
                scanning_count += 1  # Increment count when scanning behavior starts
        else:
            if is_scanning:  # If scanning just ended
                is_scanning = False  # Reset scanning flag

    return scanning_count

In [9]:
def process_file(file_path):
    global scanning_duration_df, freezing_duration_df, rearing_duration_df
    print("Processing file:", file_path)
    dlc = pd.read_csv(file_path, skiprows=1)
    result = pd.DataFrame(trim(dlc))
    result = calculate_speed(result)

    # Save the result to a new CSV file
    result.to_csv(os.path.join(velocity_folder, os.path.basename(file_path) + "-velocity_data.csv"), index=False)

    bin_scanning_duration = []

    # quantify scanning
    for i in range(len(bin_edges) - 1):
        bin_start = bin_edges[i]
        bin_end = bin_edges[i + 1]
        # Filter data for current bin
        bin_data = result[(result['Second'] >= bin_start) & (result['Second'] < bin_end)]

        scanning_duration = scanning_dur(bin_data, bin_end)

        bin_scanning_duration.append(scanning_duration)

    # Create a dictionary with scanning durations
    duration_dict = {'Number': os.path.basename(file_path)}
    duration_dict.update(zip(bin_names, bin_scanning_duration))

    # Append to respective DataFrames
    scanning_duration_df = pd.concat([scanning_duration_df, pd.DataFrame([duration_dict])], ignore_index=True)


In [10]:
import os

# List all files in the directory
file_list = os.listdir(file_path)

# Loop through each file in the directory
for filename in file_list:
    # Construct the full file path
    full_file_path = os.path.join(file_path, filename)

    # Check if the item is a file (not a directory) and ends with 'filtered.csv'
    if os.path.isfile(full_file_path) and filename.endswith('.csv') and ("velocity" not in filename):
        process_file(full_file_path)

# Replace empty (NaN) values with 0 in the DataFrames
scanning_duration_df = scanning_duration_df.fillna(0)

# Save the DataFrames to CSV files with appropriate filenames
scanning_duration_df.to_csv(os.path.join(file_path, "Hab_scan.csv"), index=False)

Processing file: /Users/elliekogan/Desktop/videos/csv/363.2_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv
Columns after removal: Index(['bodyparts', 'Nose', 'Nose.1', 'Tail_base', 'Tail_base.1', 'Frame'], dtype='object')
Number of columns after removal: 6
Processing file: /Users/elliekogan/Desktop/videos/csv/364.3_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv
Columns after removal: Index(['bodyparts', 'Nose', 'Nose.1', 'Tail_base', 'Tail_base.1', 'Frame'], dtype='object')
Number of columns after removal: 6
Processing file: /Users/elliekogan/Desktop/videos/csv/365.3_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv
Columns after removal: Index(['bodyparts', 'Nose', 'Nose.1', 'Tail_base', 'Tail_base.1', 'Frame'], dtype='object')
Number of columns after removal: 6
Processing file: /Users/elliekogan/Desktop/videos/csv/363.1_VideoDLC_HrnetW32_DLC_vids_2keyFeb27shuffle1_detector_030_snapshot_040.csv
Columns af

/var/folders/qz/pmy_02y95f9bnkfj0nt_jdx40000gn/T/ipykernel_83674/76550284.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  scanning_duration_df = scanning_duration_df.fillna(0)
